In [29]:
import sys
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd()
if (cwd / "src").exists():
    project_root = cwd
elif (cwd / "integration.py").exists():
    project_root = cwd.parent
else:
    project_root = cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
print("Project root:", project_root)

Project root: c:\Users\Acer\Documents\RAG-from-scratch\RAG-from-scratch


In [30]:
# Environment

from dotenv import load_dotenv

# LangChain core

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# Local Ollama

from langchain_ollama import ChatOllama, OllamaEmbeddings

# Data sources

from data_source.wikipedia import WikipediaSource
from data_source.web_search import WebSearchSource

# Core RAG pipeline

from src.pipeline.indexing_rag import build_vectorstore
from src.pipeline.retrieval_rag import create_retriever

# Query processing

from src.query_translation.multi_query import (
    create_multi_query_retrieval_chain,
)

# Reranking

from src.Reranking.reranking import CrossEncoderReranker

# Advanced indexing

from src.advanced_indexing.raptor import RaptorIndexer

# Advanced RAG

from src.advanced_RAG.Self_RAG.self_rag import SelfRAG
from src.advanced_RAG.Long_Context.long_context import LongContext

# Memory

from src.memory.memory import ConversationMemory

# Evaluation

from src.evaluation.rag_evaluation import RAGEvaluator

In [31]:
# Load environment variables

load_dotenv()


# Local LLM

llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
    base_url="http://127.0.0.1:11434",
)


# Local embedding model

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
    base_url="http://127.0.0.1:11434",
)

In [32]:
# Final generation prompt

generation_prompt = ChatPromptTemplate.from_template("""
You are a careful RAG assistant.

Answer the question using ONLY the provided evidence.

Evidence:
{context}

Question:
{question}

Rules:
- Do not invent facts.
- Use only the provided evidence.
- For current or time-sensitive questions, prefer the newest
  evidence with an explicit date and time.
- If sources conflict, prefer the most recent dated evidence.
- Do not use older information when newer evidence is available.
- If the evidence is insufficient, say that the information
  is not available in the retrieved evidence.
- Answer clearly and concisely.

Answer:
""")

generation_chain = generation_prompt | llm | StrOutputParser()

In [33]:
# Data sources

wikipedia = WikipediaSource(
    top_k=5,
)

web_search = WebSearchSource(
    top_k=5,
)


# Reranker

reranker = CrossEncoderReranker(
    top_k=5,
)


# Conversation memory

conversation_memory = ConversationMemory()


# Evaluator

evaluator = RAGEvaluator(
    model="llama3:latest",
    temperature=0,
)

In [34]:
# Ollama answerability check

answerability_prompt = ChatPromptTemplate.from_template("""
Determine whether you can answer the user's question reliably
using your existing knowledge.

Return ONLY one of these labels:

ANSWERABLE
NOT_ANSWERABLE

Rules:
- Return NOT_ANSWERABLE for current, live, latest, recent,
  today's, yesterday's, tomorrow's, or time-sensitive information.
- Return NOT_ANSWERABLE if you are uncertain.
- Return ANSWERABLE only when you are reasonably confident
  that your existing knowledge is sufficient.

Question:
{question}

Decision:
""")

answerability_chain = answerability_prompt | llm | StrOutputParser()

In [35]:
# User query

query = input("Ask a question: ")


# Ask Ollama first

decision = (
    answerability_chain.invoke(
        {
            "question": query,
        }
    )
    .strip()
    .upper()
)

if "NOT_ANSWERABLE" in decision:
    decision = "NOT_ANSWERABLE"

elif "ANSWERABLE" in decision:
    decision = "ANSWERABLE"

else:
    # Conservative fallback
    decision = "NOT_ANSWERABLE"

print("Ollama decision:", decision)

Ollama decision: NOT_ANSWERABLE


In [36]:
# Determine retrieval path

if decision == "ANSWERABLE":

    response = llm.invoke(query)
    answer = response.content

    print("Answered directly by Ollama.")

else:

    print("Ollama could not answer reliably.")
    print("External retrieval required.")

Ollama could not answer reliably.
External retrieval required.


In [37]:
# Detect current / time-sensitive queries


def is_current_query(query: str) -> bool:
    keywords = [
        "today",
        "current",
        "latest",
        "now",
        "live",
        "recent",
        "yesterday",
        "tomorrow",
    ]

    query_lower = query.lower()

    return any(keyword in query_lower for keyword in keywords)


current_query = is_current_query(query)

print("Current query:", current_query)

Current query: True


In [38]:
# External retrieval

documents = []

if decision == "NOT_ANSWERABLE":

    if current_query:

        # Current / time-sensitive information
        documents = web_search.retrieve(query)

        print("Source: Tavily Web Search")

        print(
            "Documents:",
            len(documents),
        )

    else:

        # Stable external information
        wikipedia_documents = wikipedia.retrieve(query)
        web_documents = web_search.retrieve(query)

        documents = wikipedia_documents + web_documents

        print(
            "Wikipedia:",
            len(wikipedia_documents),
        )

        print(
            "Web:",
            len(web_documents),
        )

        print(
            "Total:",
            len(documents),
        )

Source: Tavily Web Search
Documents: 5


In [39]:
# Deduplicate external documents

if decision == "NOT_ANSWERABLE":

    unique_documents = {}

    for document in documents:

        url = document.metadata.get("url")

        if url:
            key = url
        else:
            key = document.metadata.get("title", "") + document.page_content[:200]

        if key not in unique_documents:
            unique_documents[key] = document

    documents = list(unique_documents.values())

    print(
        "Unique documents:",
        len(documents),
    )

Unique documents: 5


In [40]:
for i, document in enumerate(documents, 1):
    print(
        i,
        document.metadata.get("title"),
        document.metadata.get("url"),
    )

1 AI-Powered Nepal Stock Market Analysis & Live NEPSE Data | NEPSE Trading https://nepsetrading.com
2 No.1 online financial portal of Nepal with a complete information of Stock market. - || ShareSansar || https://www.sharesansar.com
3 Nepali Paisa | Live Market https://nepalipaisa.com/live-market
4 merolagani - Nepal Stock Exchange (NEPSE) Live Trading Data, Live Floorsheet, Live Indices, Top Gainers, Top Losers https://merolagani.com/latestmarket.aspx
5 Live Trading - || ShareSansar || https://www.sharesansar.com/live-trading


In [41]:
# Reranking
if decision == "NOT_ANSWERABLE":
    reranked_documents = reranker.rerank_documents(
        query=query,
        documents=documents,
    )

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [42]:
from datetime import datetime


def extract_document_datetime(document):
    """Extract document datetime from metadata."""
    metadata = getattr(document, "metadata", {}) or {}

    # Try common metadata keys
    for key in ["date", "datetime", "created_at", "published_at", "timestamp"]:
        value = metadata.get(key)

        if value is not None:
            return value

    return None

In [43]:
# Inspect ranking

if decision == "NOT_ANSWERABLE":

    for i, document in enumerate(
        reranked_documents,
        1,
    ):
        print(f"\n--- Rank {i} ---")

        print(
            "Title:",
            document.metadata.get("title"),
        )

        print(
            "URL:",
            document.metadata.get("url"),
        )

        print(
            "Date:",
            extract_document_datetime(document),
        )

        print(document.page_content[:700])


--- Rank 1 ---
Title: AI-Powered Nepal Stock Market Analysis & Live NEPSE Data | NEPSE Trading
URL: https://nepsetrading.com
Date: None
See All

### NEPSE Today Full Analysis (2026-08-25): Index Movement Turnover and Tomorrow Outlook

Complete NEPSE analysis for 2026-08-25: Index 2594.27 (-5.31 pts, -0.20%). 20 up, 20 down. RSI 0 Buy. MACD 0% positive (market-wide). Tomorrow: cautious.

· Aug 25, 2026### NEPSE Daily Closing Nepal (2026-08-25): Market Strength Weakness and Trading Strategy [...] · Aug 25, 2026### NEPSE Market Update (2026-08-25): Sector Rotation and Key Movers Today

HYDRO POWER saw mixed action with 8 gainers and 8 losers today. NEPSE -5.31 pts to 2594.27. Banking -0.04%, Hydropower -0.43%. Sector rotation map and key movers.

· Aug 25, 2026### NEPSE Today Closing (2026-08-25): Profit Booking or Fresh Buying

--- Rank 2 ---
Title: No.1 online financial portal of Nepal with a complete information of Stock market. - || ShareSansar ||
URL: https://www.sharesansar.com
Dat

In [44]:
# Build vector store

vectorstore = None
retriever = None
multi_query_retriever = None

if decision == "NOT_ANSWERABLE" and not current_query:

    vectorstore = build_vectorstore(
        documents=documents,
        embedding_model=embeddings,
        batch_size=32,
    )

    print("Vector store created.")

In [45]:
# Create retriever

if decision == "NOT_ANSWERABLE" and not current_query:

    retriever = create_retriever(
        vectorstore=vectorstore,
        k=5,
    )

In [46]:
# Multi-Query retrieval

if decision == "NOT_ANSWERABLE" and not current_query:

    multi_query_retriever = create_multi_query_retrieval_chain(
        retriever=retriever,
        llm=llm,
    )

    retrieved_documents = multi_query_retriever.invoke(query)

    print(
        "Retrieved documents:",
        len(retrieved_documents),
    )

In [47]:
# Rerank stable RAG results

if decision == "NOT_ANSWERABLE" and not current_query:

    reranked_documents = reranker.rerank_documents(
        query=query,
        documents=retrieved_documents,
        freshness_aware=False,
    )

    print(
        "Stable RAG reranked documents:",
        len(reranked_documents),
    )

In [48]:
# RAPTOR

raptor_leaf = []
raptor_clusters = []

if decision == "NOT_ANSWERABLE" and not current_query:

    raptor = RaptorIndexer(
        llm=llm,
        embeddings=embeddings,
        n_clusters=3,
    )

    raptor.build_tree(
        documents=documents,
    )

    raptor_results = raptor.retrieve(
        query=query,
        k=3,
    )

    raptor_leaf = raptor_results.get(
        "leaf",
        [],
    )

    raptor_clusters = raptor_results.get(
        "clusters",
        [],
    )

    print(
        "RAPTOR leaf results:",
        len(raptor_leaf),
    )

    print(
        "RAPTOR cluster results:",
        len(raptor_clusters),
    )

In [49]:
# Self-RAG

self_rag_answer = ""

if decision == "NOT_ANSWERABLE" and not current_query:

    self_rag = SelfRAG(
        llm=llm,
        retriever=retriever,
    )

    self_rag_answer = self_rag.invoke(
        query,
        max_retries=2,
    )

    print("Self-RAG completed.")

In [50]:
# Long-context processing

long_context_answer = ""

if decision == "NOT_ANSWERABLE" and not current_query:

    long_context = LongContext(
        model="llama3:latest",
    )

    long_context_result = long_context.run(
        documents=reranked_documents,
        query=query,
        compress=True,
    )

    if isinstance(
        long_context_result,
        dict,
    ):
        long_context_answer = long_context_result.get(
            "answer",
            "",
        )
    else:
        long_context_answer = str(long_context_result)

In [51]:
# Build final context

context = ""

if decision == "NOT_ANSWERABLE":

    reranked_context = "\n\n".join(
        document.page_content for document in reranked_documents
    )

    if current_query:

        # Current/live queries:
        # Only use the freshness-aware web evidence.

        context = reranked_context

    else:

        # Stable external RAG:
        # Combine the advanced retrieval evidence.

        context_parts = [
            reranked_context,
        ]

        if raptor_leaf:
            context_parts.append(
                "RAPTOR Leaf Evidence:\n"
                + "\n\n".join(str(item) for item in raptor_leaf)
            )

        if raptor_clusters:
            context_parts.append(
                "RAPTOR Cluster Evidence:\n"
                + "\n\n".join(str(item) for item in raptor_clusters)
            )

        if self_rag_answer:
            context_parts.append("Self-RAG Evidence:\n" + self_rag_answer)

        if long_context_answer:
            context_parts.append("Long-Context Evidence:\n" + long_context_answer)

        context = "\n\n".join(context_parts)

    print(
        "Final context length:",
        len(context),
    )

Final context length: 6237


In [52]:
# Final generation

if decision == "NOT_ANSWERABLE":

    answer = generation_chain.invoke(
        {
            "context": context,
            "question": query,
        }
    )

In [53]:
# Direct Ollama answer

if decision == "ANSWERABLE":

    response = llm.invoke(query)

    answer = response.content

In [54]:
# Store conversation memory

conversation_memory.add_message(HumanMessage(content=query))

conversation_memory.add_message(AIMessage(content=answer))

In [55]:
# Evaluation

if decision == "ANSWERABLE":

    answer_relevance = evaluator.evaluate_answer_relevance(
        question=query,
        answer=answer,
    )

    evaluation_results = {
        "answer_relevance": answer_relevance,
    }

else:

    context_relevance = evaluator.evaluate_context_relevance(
        question=query,
        context=context,
    )

    faithfulness = evaluator.evaluate_faithfulness(
        context=context,
        answer=answer,
    )

    answer_relevance = evaluator.evaluate_answer_relevance(
        question=query,
        answer=answer,
    )

    evaluation_results = {
        "context_relevance": context_relevance,
        "faithfulness": faithfulness,
        "answer_relevance": answer_relevance,
    }

print("Evaluation:")
for metric, value in evaluation_results.items():
    print(f"{metric}: {value}")

Evaluation:
context_relevance: RELEVANT

The retrieved context contains information about the NEPSE Index, including its current value, movement, and analysis. The context also provides information about sector rotation, key movers, and trading strategies, which are all relevant to the question "today nepse index".
faithfulness: FAITHFUL

The answer is fully supported by the provided context. The text explicitly states that the NEPSE Index closed at 2,594.27 (-5.31 points, -0.20%) on August 25, 2026.
answer_relevance: RELEVANT


In [56]:
# Final output

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)

print(answer)

print("=" * 60)

print(
    "\nDecision:",
    decision,
)

print(
    "Mode:",
    (
        "Direct Ollama"
        if decision == "ANSWERABLE"
        else ("Current Web RAG" if current_query else "Advanced RAG")
    ),
)


FINAL ANSWER
According to the provided evidence, the NEPSE Index closed at 2,594.27 (-5.31 points, -0.20%) on August 25, 2026.

Decision: NOT_ANSWERABLE
Mode: Current Web RAG
